# H-Ghost: real-data Falcon-H1 TPU smoke test

This is a deliberately small continued-pretraining gate: the published 91M base checkpoint, real curated corpus tokens, EasyDeL's Falcon-H1 implementation, BF16, full layer rematerialization, and eight-way data parallelism. It performs four optimizer steps and writes a machine-readable report.


In [ ]:
from __future__ import annotations

import glob
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys
import time

os.environ.setdefault('EASYDEL_AUTO', '1')
os.environ.setdefault('ENABLE_DISTRIBUTED_INIT', '0')
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
os.environ.setdefault('JAX_COMPILATION_CACHE_DIR', '/kaggle/working/jax-cache')
os.environ.setdefault('PIP_DISABLE_PIP_VERSION_CHECK', '1')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', 'easydel==0.3.0', 'pillow'],
    check=True,
)

import jax
import jax.numpy as jnp
import numpy as np
from datasets import Dataset
import easydel as ed

def exactly_one(pattern: str) -> Path:
    matches = [Path(p) for p in glob.glob(pattern, recursive=True)]
    if len(matches) != 1:
        raise RuntimeError(f'Expected one match for {pattern!r}, found {matches}')
    return matches[0]

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

hardware = {
    'jax': jax.__version__,
    'backend': jax.default_backend(),
    'device_count': jax.device_count(),
    'devices': [str(device) for device in jax.devices()],
}
print(json.dumps({'event': 'hardware', **hardware}), flush=True)
if hardware['backend'] != 'tpu' or hardware['device_count'] != 8:
    raise RuntimeError('This gate requires the complete TPU v5e-8 slice')

base_manifest_path = exactly_one('/kaggle/input/**/preflight-manifest.json')
corpus_report_path = exactly_one('/kaggle/input/**/validation-report.json')
base_root = base_manifest_path.parent
corpus_root = corpus_report_path.parent
base_manifest = json.loads(base_manifest_path.read_text())
corpus_report = json.loads(corpus_report_path.read_text())
model_path = base_root / 'model.safetensors'
train_path = corpus_root / 'train.bin'

expected_model_sha = base_manifest['files']['model.safetensors']['sha256']
expected_train_sha = corpus_report['splits']['train']['sha256']
actual_model_sha = sha256(model_path)
actual_train_sha = sha256(train_path)
if actual_model_sha != expected_model_sha:
    raise RuntimeError(f'Model checksum mismatch: {actual_model_sha}')
if actual_train_sha != expected_train_sha:
    raise RuntimeError(f'Train checksum mismatch: {actual_train_sha}')

sequence_length = 128
global_batch = 8
steps = 4
dataset_rows = global_batch * steps
token_stream = np.memmap(train_path, mode='r', dtype='<u2')
needed = dataset_rows * sequence_length
if token_stream.size < needed:
    raise RuntimeError(f'Corpus has only {token_stream.size} tokens; need {needed}')
sequences = np.asarray(token_stream[:needed], dtype=np.int32).reshape(dataset_rows, sequence_length)
dataset = Dataset.from_dict({
    'input_ids': [row.tolist() for row in sequences],
    'attention_mask': [[1] * sequence_length for _ in range(dataset_rows)],
})
print(json.dumps({
    'event': 'real_dataset',
    'rows': dataset_rows,
    'sequence_length': sequence_length,
    'first_tokens': sequences[0, :16].tolist(),
    'train_sha256': actual_train_sha,
}), flush=True)

load_started = time.time()
model = ed.AutoEasyDeLModelForCausalLM.from_pretrained(
    str(base_root),
    from_torch=True,
    auto_shard_model=True,
    dtype=jnp.bfloat16,
    param_dtype=jnp.bfloat16,
    precision=jax.lax.Precision.DEFAULT,
    sharding_axis_dims=(8, 1, 1, 1, 1),
    config_kwargs=ed.EasyDeLBaseConfigDict(
        freq_max_position_embeddings=sequence_length,
        mask_max_position_embeddings=sequence_length,
        attn_mechanism=ed.AttentionMechanisms.VANILLA,
        attn_dtype=jnp.bfloat16,
        gradient_checkpointing=ed.EasyDeLGradientCheckPointers.NOTHING_SAVEABLE,
    ),
    verbose=True,
)
print(json.dumps({
    'event': 'model_loaded',
    'seconds': time.time() - load_started,
    'parameters': base_manifest['parameter_count'],
    'model_sha256': actual_model_sha,
    'mesh': str(model.mesh),
}), flush=True)

output_root = Path('/kaggle/working/easydel-real-cpt-smoke')
arguments = ed.TrainingArguments(
    save_directory=str(output_root),
    num_train_epochs=1,
    max_training_steps=steps,
    total_batch_size=global_batch,
    gradient_accumulation_steps=1,
    learning_rate=1e-6,
    weight_decay=0.01,
    clip_grad=1.0,
    max_length=sequence_length,
    log_steps=1,
    report_steps=1,
    log_grad_norms=True,
    shuffle_train_dataset=False,
    use_grain=True,
    dataloader_num_workers=0,
    use_wandb=False,
    use_esurge_generation=False,
    progress_bar_type='json',
    low_mem_usage=True,
    track_memory=False,
    resume_if_possible=False,
    save_tpu_preemption_checkpoints=False,
    save_optimizer_state=False,
    do_last_save=False,
    verbose=True,
)
trainer = ed.Trainer(arguments=arguments, model=model, dataset_train=dataset)
train_started = time.time()
result = trainer.train()
train_seconds = time.time() - train_started
completed_steps = int(np.asarray(jax.device_get(result.state.step)))
if completed_steps != steps:
    raise RuntimeError(f'Expected {steps} optimizer steps, got {completed_steps}')
visited_tokens = completed_steps * global_batch * sequence_length
report = {
    'ok': True,
    **hardware,
    'runtime': 'EasyDeL 0.3.0',
    'parameters': base_manifest['parameter_count'],
    'pretrained_checkpoint': base_manifest['model'],
    'model_sha256': actual_model_sha,
    'train_sha256': actual_train_sha,
    'real_corpus': True,
    'sequence_length': sequence_length,
    'global_batch': global_batch,
    'steps': completed_steps,
    'visited_tokens': visited_tokens,
    'train_seconds_including_compile': train_seconds,
    'end_to_end_tokens_per_second': visited_tokens / train_seconds,
    'gradient_checkpointing': 'nothing_saveable',
    'checkpoint_saved': False,
}
Path('/kaggle/working/tpu-easydel-real-cpt-smoke-report.json').write_text(
    json.dumps(report, indent=2) + '\n'
)
print(json.dumps({'event': 'complete', **report}), flush=True)
print('TPU_V5E8_EASYDEL_REAL_CPT_SMOKE_OK', flush=True)
